# Chapter 2 — Accord Combinations

Publication-oriented analysis, summary tables and visualizations for *Fragrance Accord Intelligence*.

This notebook prepares the Chapter 2 outputs for ordered accord dyads and triads. It reuses the source definitions and median-based segmentation logic established in the earlier analysis, while applying the shared publication design system.

## Figure and table inventory

| Item | Title |
|---|---|
| Figure 2.1 | Top 20 Ordered Accord Pairs |
| Figure 2.2 | Top 20 Ordered Accord Triads |
| Figure 2.3 | Dyad Attention–Rating Segmentation |
| Figure 2.4 | Triad Attention–Rating Segmentation |
| Figures 2.5–2.8 | Dyad Segment Profiles |
| Figures 2.9–2.12 | Triad Segment Profiles |
| Table 2.1 | Dyad Segment Summary |
| Table 2.2 | Triad Segment Summary |

## Analytical conventions

- Dyads preserve the first and second accord positions; reversed pairs remain distinct.
- Triads preserve the first, second and third accord positions.
- Dyad aggregation retains combinations with at least 30 fragrances; segmentation requires at least 100.
- Triad segmentation requires at least 50 fragrances.
- Attention is measured as mean community votes per fragrance.
- Ratings are platform-user evaluations, not objective quality measures.
- Segment thresholds are calculated independently for dyads and triads.
- Median ties are assigned to the higher side using the `>=` rule.
- Segment-specific charts rank combinations by the share of associated fragrances rated 4.5 or above. This is a secondary descriptive measure, not the basis for segment membership.
- Fragrance counts across combinations are assignments and should not be interpreted as unique fragrance totals.

## 0. Setup and source data

Load the cleaned Fragrantica source table and prepare the rated-fragrance dataset. Accord parsing and positional extraction follow the established Notebook 10 logic.

In [0]:
# Install the two publication-oriented libraries used in this notebook.
# Run this cell once per Databricks cluster/environment.
%pip install lets-plot great-tables
%pip install adjustText

In [0]:
# ============================================================
# NOTEBOOK 16 — CHAPTER 1 DOMINANT ACCORD VISUALIZATIONS
# ============================================================

from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import base64
import io

from great_tables import GT, style, loc
from lets_plot import (
    LetsPlot, ggplot, aes, geom_bar, geom_text, coord_flip,
    labs, theme, theme_minimal, element_text, element_line,
    element_rect, element_blank, scale_fill_manual, scale_y_continuous,
    ggsave
)

# Configure Lets-Plot for notebook rendering.
LetsPlot.setup_html()

# Source table
SOURCE_TABLE = "main.fragrance_analytics.fragrantica_perfumes"
df_perfumes = spark.table(SOURCE_TABLE)

# Parse main_accords using the same logic as Notebook 08
df_accords = df_perfumes.withColumn(
    "accords",
    F.from_json(
        F.regexp_replace(
            F.regexp_replace(
                F.col("main_accords"),
                "'",
                '"'
            ),
            r"\bNone\b",
            "null"
        ),
        "array<string>"
    )
)

# One row per fragrance–accord combination.
# Null and empty accord labels are excluded, as in Notebook 08.
df_fragrance_accords = (
    df_accords
    .select(
        "id",
        "name",
        "gender",
        "rating_value",
        "rating_count",
        F.explode("accords").alias("accord")
    )
    .withColumn("accord", F.lower(F.trim(F.col("accord"))))
    .filter(
        F.col("accord").isNotNull()
        & (F.col("accord") != "")
    )
)

print(f"Source table: {SOURCE_TABLE}")
print(f"Source records: {df_perfumes.count():,}")
print(f"Distinct fragrance IDs: {df_perfumes.select('id').distinct().count():,}")
print(f"Distinct valid accord labels: {df_fragrance_accords.select('accord').distinct().count():,}")

## 0.1 Global visualisation standards

The shared publication design system below is reused from the earlier book notebooks. It preserves the warm-ivory background, typography, palette, figure headers, table styling and rendering helpers.

In [0]:
# ============================================================
# GLOBAL VISUALISATION DESIGN SYSTEM — ALL TABLES AND CHARTS
# ============================================================

# Single source of truth for colours, typography and sizing.
BOOK_STYLE = {
    "background": "#F7F3EC",
    "surface": "#F7F3EC",
    "text": "#302824",
    "muted": "#756B64",
    "accent": "#A47C59",
    "accent_light": "#D8C3AD",
    "rule": "#E5DDD2",
    "grid": "#E9E2D8",
    "men": "#687F8C",
    "women": "#B47C83",
    "unisex": "#A58B68",
    "other": "#B9B2AC"
}

BOOK_TYPOGRAPHY = {
    "display": "DejaVu Serif",
    "body": "DejaVu Sans"
}

BOOK_FIGURE_CHART_TOP_IN = 1.45  # Fixed physical distance from figure top to chart area

# Shared horizontal-bar layout: chart body starts closer to the left edge.
BOOK_HEADER_X = 0.06
BOOK_BAR_LEFT = 0.15

def apply_book_bar_layout(fig, bottom=0.10, right=0.96):
    """Apply the same left plot margin to every horizontal bar chart."""
    fig.subplots_adjust(
        left=BOOK_BAR_LEFT,
        right=right,
        top=1 - BOOK_FIGURE_CHART_TOP_IN / fig.get_figheight(),
        bottom=bottom
    )

BOOK_SIZES = {
    "title": 22,
    "label": 12,                 # shared readable axis/category/value labels in points
    "bar_value_mm": 4.2,         # Lets-Plot geom_text uses millimetres; approximately 12 pt
    "bar_width": 0.30,           # consistent slim bar thickness across all horizontal charts
    "title_pt": 22,                # Shared figure title size in points
    "table_body_px": "15px",
    "table_title_px": "26px",
    "table_subtitle_px": "15px",
    "section_pt": 10,
    "subtitle_pt": 13,              # Readable shared figure subtitle size in points
    "category_pt": 12,              # Shared category and axis-label size in points
    "value_pt": 12,                 # Match category labels and data-label typography
    "footer_pt": 10,
    "matrix_value_pt": 24
}

plt.rcParams.update({
    "figure.facecolor": BOOK_STYLE["background"],
    "axes.facecolor": BOOK_STYLE["background"],
    "savefig.facecolor": BOOK_STYLE["background"],
    "figure.edgecolor": BOOK_STYLE["background"],
    "text.color": BOOK_STYLE["text"],
    "axes.labelcolor": BOOK_STYLE["text"],
    "xtick.color": BOOK_STYLE["text"],
    "ytick.color": BOOK_STYLE["text"],
    "font.family": BOOK_TYPOGRAPHY["body"],
    "font.size": BOOK_SIZES["label"],
    "axes.titlesize": BOOK_SIZES["title"],
    "axes.titleweight": "bold",
    "axes.titlelocation": "center",
    "axes.labelsize": BOOK_SIZES["label"],
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

def make_book_table(data, title, subtitle=None):
    """Build a table using the global ivory, typography and title rules."""
    table = (
        GT(data)
        .tab_header(title=title, subtitle=subtitle)
        .tab_options(
            table_background_color=BOOK_STYLE["background"],
            heading_background_color=BOOK_STYLE["background"],
            column_labels_background_color=BOOK_STYLE["background"],
            table_font_names=[BOOK_TYPOGRAPHY["body"], "Arial", "sans-serif"],
            table_font_size=BOOK_SIZES["table_body_px"],
            table_font_color=BOOK_STYLE["text"],
            heading_align="left",
            heading_title_font_size=BOOK_SIZES["table_title_px"],
            heading_subtitle_font_size=BOOK_SIZES["table_subtitle_px"],
            heading_padding="14px",
            column_labels_font_weight="bold",
            column_labels_padding="10px",
            table_body_hlines_color=BOOK_STYLE["grid"],
            table_body_hlines_width="1px",
            data_row_padding="11px"
        )
    )
    table = table.tab_style(
        style.fill(color=BOOK_STYLE["background"]),
        locations=[loc.body(), loc.column_header(), loc.header()]
    )
    table = table.tab_style(
        style.text(font=BOOK_TYPOGRAPHY["display"], size=BOOK_SIZES["table_title_px"], weight="bold", color=BOOK_STYLE["text"]),
        locations=[loc.title()]
    )
    table = table.tab_style(
        style.text(font=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["table_subtitle_px"], color=BOOK_STYLE["muted"]),
        locations=[loc.subtitle()]
    )
    return table

def add_book_figure_header(fig, title, subtitle, section="FRAGRANCE INTELLIGENCE  /  CHAPTER 1"):
    """Apply identical header coordinates, typography and physical spacing to every figure."""
    figure_height = fig.get_figheight()
    section_y = 1 - 0.203 / figure_height
    title_y = 1 - 0.435 / figure_height
    subtitle_y = 1 - 0.841 / figure_height

    fig.text(
        BOOK_HEADER_X, section_y, section,
        color=BOOK_STYLE["accent"],
        fontsize=BOOK_SIZES["section_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, title_y, title,
        color=BOOK_STYLE["text"],
        fontsize=BOOK_SIZES["title_pt"],
        fontfamily=BOOK_TYPOGRAPHY["display"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, subtitle_y, subtitle,
        color=BOOK_STYLE["muted"],
        fontsize=BOOK_SIZES["subtitle_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        ha="left",
        va="top"
    )

def show_book_table(table):
    html = table.as_raw_html(all_important=True)
    displayHTML(f'<div class="nodarkmode">{html}</div>')

def show_book_plot(plot):
    html = plot.to_html(iframe=True)
    displayHTML(f'<div class="nodarkmode">{html}</div>')

def show_book_matplotlib(fig):
    buffer = io.BytesIO()
    # Keep the full figure canvas so titles retain the same visible left position
    # across figures with different tick-label widths.
    fig.savefig(
        buffer, format="png", dpi=240, bbox_inches=None,
        facecolor=BOOK_STYLE["background"],
        edgecolor=BOOK_STYLE["background"]
    )
    image_b64 = base64.b64encode(buffer.getvalue()).decode("ascii")
    displayHTML(
        '<div class="nodarkmode">'
        f'<img src="data:image/png;base64,{image_b64}" '
        'style="display:block;max-width:100%;height:auto;'
        f'background:{BOOK_STYLE["background"]};">'
        '</div>'
    )

BOOK_PLOT_THEME = (
    theme_minimal()
    + theme(
        plot_title=element_text(
            family=BOOK_TYPOGRAPHY["display"],
            size=BOOK_SIZES["title_pt"],
            face="bold",
            color=BOOK_STYLE["text"],
            hjust=0.5
        ),
        plot_subtitle=element_text(
            family=BOOK_TYPOGRAPHY["body"],
            size=BOOK_SIZES["subtitle_pt"],
            color=BOOK_STYLE["muted"],
            hjust=0.5
        ),
        axis_title=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["muted"]
        ),
        axis_text=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_text_y=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_ticks=element_blank(),
        panel_grid_major_y=element_blank(),
        panel_grid_major_x=element_line(color=BOOK_STYLE["grid"], size=0.6),
        panel_grid_minor=element_blank(),
        panel_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        plot_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        legend_position="none"
    )
)

print("Global visualisation standards are ready.")

## 0.2 Shared rated-fragrance accord data

Prepare the fragrance-level dataset once for both dyad and triad analyses. Preserve the original first, second and third accord positions, and validate the one-row-per-fragrance assumption before aggregation.

In [ ]:
# ============================================================
# CHAPTER 2 — SHARED RATED-FRAGRANCE ACCORD DATA
# Shared by ordered dyad and triad analysis.
# ============================================================

from pathlib import Path
from pyspark.sql import functions as F

CHAPTER_2_OUTPUT_DIR = Path(
    "/Volumes/main/fragrance_analytics/reference_data/visualizations/"
)
CHAPTER_2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Recreate the valid-accord flag using Notebook 10's definition.
accords_clean_ch2 = F.trim(F.col("main_accords"))

valid_accords_ch2 = (
    F.col("main_accords").isNotNull()
    & (accords_clean_ch2 != "")
    & (~F.lower(accords_clean_ch2).isin("null", "none", "nan", "n/a", "[]"))
)

fragrantica_rating_base_ch2 = df_perfumes.withColumn(
    "has_valid_accords",
    valid_accords_ch2
)

# Parse accord arrays while preserving their original order.
accord_array_ch2 = F.from_json(
    F.regexp_replace(F.col("main_accords"), "'", '"'),
    "array<string>"
)

rating_accords_ch2 = (
    fragrantica_rating_base_ch2
    .filter(
        F.col("has_valid_accords")
        & F.col("rating_value").isNotNull()
    )
    .withColumn("accords", accord_array_ch2)
    .withColumn(
        "accords",
        F.transform("accords", lambda x: F.lower(F.trim(x)))
    )
)

# Extract first, second and third accord positions without reordering.
for position, column_name in [
    (0, "dominant_accord"),
    (1, "second_accord"),
    (2, "third_accord")
]:
    rating_accords_ch2 = rating_accords_ch2.withColumn(
        column_name,
        F.when(
            (F.size("accords") > position)
            & F.col("accords")[position].isNotNull()
            & (F.trim(F.col("accords")[position]) != ""),
            F.col("accords")[position]
        )
    )

rating_accords_ch2 = rating_accords_ch2.select(
    "id",
    "name",
    "gender",
    "rating_value",
    "rating_count",
    "dominant_accord",
    "second_accord",
    "third_accord"
)

# Validate the one-row-per-fragrance assumption before either aggregation.
source_rows_ch2 = rating_accords_ch2.count()
distinct_ids_ch2 = rating_accords_ch2.select("id").distinct().count()
assert source_rows_ch2 == distinct_ids_ch2, (
    f"Duplicate fragrance records detected: {source_rows_ch2:,} rows "
    f"versus {distinct_ids_ch2:,} unique IDs."
)
print(f"Rated fragrances with valid accord data: {source_rows_ch2:,}")
print(f"Distinct fragrance IDs: {distinct_ids_ch2:,}")
print(f"Chapter 2 output directory: {CHAPTER_2_OUTPUT_DIR}")

## 2.1 Dyad aggregation

Reproduce the ordered pair extraction and aggregation from Notebook 10 using the same fragrance-level source and first/second accord positions. Objects use a `_ch2` suffix to avoid collisions.

In [ ]:
# ============================================================
# CHAPTER 2 — ORDERED DYAD DATA PREPARATION
# ============================================================

MIN_PAIR_FRAGRANCES_CH3 = 30
MIN_SEGMENT_PAIR_FRAGRANCES_CH3 = 100

accord_pair_stats_ch2 = (
    rating_accords_ch2
    .filter(
        F.col("id").isNotNull()
        & F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
        & F.col("rating_value").isNotNull()
        & F.col("rating_count").isNotNull()
        & (F.col("rating_count") >= 0)
    )
    .groupBy("dominant_accord", "second_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.sum("rating_count").alias("total_votes"),
        F.avg("rating_count").alias("mean_votes"),
        F.expr("percentile_approx(rating_count, 0.5)").alias("median_votes"),
        F.avg("rating_value").alias("mean_rating"),
        F.expr("percentile_approx(rating_value, 0.5)").alias("median_rating"),
        F.avg(F.when(F.col("rating_value") >= 4.5, 1.0).otherwise(0.0)).alias("share_rated_4_5_plus")
    )
    .filter(F.col("fragrance_count") >= MIN_PAIR_FRAGRANCES_CH3)
    .withColumn("accord_pair", F.concat_ws(" + ", F.initcap("dominant_accord"), F.initcap("second_accord")))
)

eligible_pairs_ch2 = accord_pair_stats_ch2.filter(
    F.col("fragrance_count") >= MIN_SEGMENT_PAIR_FRAGRANCES_CH3
)

print(f"Ordered pairs with n >= {MIN_PAIR_FRAGRANCES_CH3}: {accord_pair_stats_ch2.count():,}")
print(f"Eligible ordered pairs for segmentation (n >= {MIN_SEGMENT_PAIR_FRAGRANCES_CH3}): {eligible_pairs_ch2.count():,}")

display(
    eligible_pairs_ch2.orderBy(F.desc("fragrance_count"), F.desc("mean_rating"))
    .select("accord_pair", "fragrance_count", "mean_votes", "mean_rating", "share_rated_4_5_plus")
    .limit(20)
)

In [ ]:
# ============================================================
# TABLE 2.1 — ORDERED DYAD SEGMENTATION SUMMARY
# ============================================================

pairs_pdf_ch2 = eligible_pairs_ch2.toPandas().copy()
for metric in ["fragrance_count", "mean_votes", "mean_rating", "share_rated_4_5_plus"]:
    pairs_pdf_ch2[metric] = pd.to_numeric(pairs_pdf_ch2[metric], errors="coerce")

pairs_pdf_ch2 = pairs_pdf_ch2.dropna(subset=[
    "accord_pair", "fragrance_count", "mean_votes", "mean_rating", "share_rated_4_5_plus"
]).copy()

assert len(pairs_pdf_ch2) == 147, (
    f"Expected 147 eligible ordered pairs; found {len(pairs_pdf_ch2):,}. "
    "Review the source snapshot and eligibility rules before publication."
)

popularity_median_pairs_ch2 = pairs_pdf_ch2["mean_votes"].median()
rating_median_pairs_ch2 = pairs_pdf_ch2["mean_rating"].median()

def assign_pair_segment_ch2(row):
    high_attention = row["mean_votes"] >= popularity_median_pairs_ch2
    high_rating = row["mean_rating"] >= rating_median_pairs_ch2
    if high_attention and high_rating:
        return "Standout Profiles"
    if not high_attention and high_rating:
        return "Hidden Favourites"
    if high_attention and not high_rating:
        return "Widely Explored"
    return "Quiet Profiles"

pairs_pdf_ch2["segment"] = pairs_pdf_ch2.apply(assign_pair_segment_ch2, axis=1)
segment_order_pairs_ch2 = [
    "Standout Profiles", "Hidden Favourites", "Widely Explored", "Quiet Profiles"
]
segment_colors_pairs_ch2 = {
    "Standout Profiles": BOOK_STYLE["accent"],
    "Hidden Favourites": BOOK_STYLE["women"],
    "Widely Explored": BOOK_STYLE["men"],
    "Quiet Profiles": BOOK_STYLE["unisex"]
}

assert abs(popularity_median_pairs_ch2 - 220.1) < 0.2, (
    f"Unexpected pair attention median: {popularity_median_pairs_ch2:.3f}"
)
assert abs(rating_median_pairs_ch2 - 2.975) < 0.002, (
    f"Unexpected pair rating median: {rating_median_pairs_ch2:.4f}"
)

print(f"Eligible ordered pairs: {len(pairs_pdf_ch2):,}")
print(f"Median mean votes per fragrance: {popularity_median_pairs_ch2:,.1f}")
print(f"Median mean fragrance rating: {rating_median_pairs_ch2:.3f}")

pair_segment_summary_ch2 = (
    pairs_pdf_ch2.groupby("segment", observed=False)
    .agg(
        pair_count=("accord_pair", "count"),
        fragrance_assignments=("fragrance_count", "sum"),
        median_fragrance_count=("fragrance_count", "median"),
        mean_votes_per_pair=("mean_votes", "mean"),
        mean_rating_per_pair=("mean_rating", "mean"),
        mean_pair_share_rated_4_5_plus=("share_rated_4_5_plus", "mean")
    )
    .reindex(segment_order_pairs_ch2).reset_index()
)

pair_segment_display_ch2 = pair_segment_summary_ch2.copy()
pair_segment_display_ch2["median_fragrance_count"] = pair_segment_display_ch2["median_fragrance_count"].round(0).astype("Int64")
pair_segment_display_ch2["mean_votes_per_pair"] = pair_segment_display_ch2["mean_votes_per_pair"].round(1)
pair_segment_display_ch2["mean_rating_per_pair"] = pair_segment_display_ch2["mean_rating_per_pair"].round(3)
pair_segment_display_ch2["mean_pair_share_rated_4_5_plus"] = pair_segment_display_ch2["mean_pair_share_rated_4_5_plus"].map(
    lambda x: f"{x:.1%}" if pd.notna(x) else ""
)

pair_segment_table_ch2 = make_book_table(
    pair_segment_display_ch2.rename(columns={
        "segment": "Segment",
        "pair_count": "Ordered dyads",
        "fragrance_assignments": "Fragrances represented",
        "median_fragrance_count": "Median fragrances per dyad",
        "mean_votes_per_pair": "Mean votes per fragrance",
        "mean_rating_per_pair": "Mean rating",
        "mean_pair_share_rated_4_5_plus": "Mean dyad share rated 4.5+"
    }),
    title="Ordered Dyad Segmentation Summary",
    subtitle="Segment means are unweighted across dyads; fragrance counts are summed group-level dyad counts."
)
show_book_table(pair_segment_table_ch2)

display(pairs_pdf_ch2[
    ["accord_pair", "fragrance_count", "mean_votes", "mean_rating", "share_rated_4_5_plus", "segment"]
].sort_values("fragrance_count", ascending=False).head(20))

## 2.2 Figure 2.1 — Most Frequent Ordered Accord Dyads

The chart ranks the 20 most frequent ordered dyads by the number of fragrances assigned to each combination. Accord order is retained in every label.

In [ ]:
# ============================================================
# FIGURE 2.1 — MOST FREQUENT ORDERED DYADS
# ============================================================

def format_pair_label_ch2(value):
    return str(value)

plot_pairs_frequency_ch2 = (
    pairs_pdf_ch2.nlargest(20, "fragrance_count")
    .sort_values("fragrance_count", ascending=True).copy()
)
fig_3_1_dyads, ax = plt.subplots(figsize=(15, 10), dpi=180)
fig_3_1_dyads.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])
y = np.arange(len(plot_pairs_frequency_ch2))
bars = ax.barh(y, plot_pairs_frequency_ch2["fragrance_count"],
               color=BOOK_STYLE["accent"], edgecolor="none", height=0.68, zorder=3)
ax.set_yticks(y)
ax.set_yticklabels(plot_pairs_frequency_ch2["accord_pair"],
                   fontsize=BOOK_SIZES["category_pt"], color=BOOK_STYLE["text"])
max_value = float(plot_pairs_frequency_ch2["fragrance_count"].max())
for bar, value in zip(bars, plot_pairs_frequency_ch2["fragrance_count"]):
    ax.text(float(value) + max_value * 0.012, bar.get_y() + bar.get_height()/2,
            f"{int(value):,}", va="center", ha="left",
            fontsize=BOOK_SIZES["value_pt"], color=BOOK_STYLE["text"])
ax.set_xlim(0, max_value * 1.14)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="both", length=0, labelsize=BOOK_SIZES["category_pt"], pad=7)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.grid(axis="x", color=BOOK_STYLE["grid"], linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
apply_book_bar_layout(fig_3_1_dyads, bottom=0.08, right=0.96)
add_book_figure_header(
    fig_3_1_dyads,
    "The Most Frequent Ordered Accord Dyads",
    "Twenty most frequent combinations; n denotes the number of fragrances",
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)
fig_3_1_dyads.savefig(
    CHAPTER_2_OUTPUT_DIR / "figure_2_1_top_20_ordered_dyads.png",
    dpi=300, bbox_inches=None, facecolor=BOOK_STYLE["background"]
)
show_book_matplotlib(fig_3_1_dyads)
plt.close(fig_3_1_dyads)
print(f"Saved: {CHAPTER_2_OUTPUT_DIR / 'figure_2_1_top_20_ordered_dyads.png'}")

## 2.3 Figure 2.2 — Community Attention and Mean Rating Across Ordered Dyads

The bubble chart positions all 147 eligible ordered dyads by mean community votes per fragrance and mean fragrance rating. Bubble size represents the number of fragrances in the dyad. Dashed reference lines mark medians across the eligible pair population. Up to 35 labels are selected for readability; they do not define the analytical population.

In [ ]:
# ============================================================
# FIGURE 2.2 — ORDERED DYAD ATTENTION / RATING SEGMENTATION
# ============================================================

N_LARGEST_PAIRS_CH3 = 10
N_EXTREMES_PER_PAIR_METRIC_CH3 = 10
MAX_PAIR_LABELS_CH3 = 35
min_pair_bubble_size_ch2, max_pair_bubble_size_ch2 = 55, 950
pair_counts_ch2 = pairs_pdf_ch2["fragrance_count"].astype(float)

if pair_counts_ch2.max() == pair_counts_ch2.min():
    pairs_pdf_ch2["bubble_size"] = min_pair_bubble_size_ch2
else:
    pairs_pdf_ch2["bubble_size"] = (
        min_pair_bubble_size_ch2
        + ((pair_counts_ch2-pair_counts_ch2.min())/(pair_counts_ch2.max()-pair_counts_ch2.min()))**0.5
        * (max_pair_bubble_size_ch2-min_pair_bubble_size_ch2)
    )

largest_pairs_ch2 = pairs_pdf_ch2.nlargest(N_LARGEST_PAIRS_CH3, "fragrance_count")
pair_labels_ch2 = pd.concat([
    pairs_pdf_ch2.nlargest(N_EXTREMES_PER_PAIR_METRIC_CH3, "mean_votes"),
    pairs_pdf_ch2.nsmallest(N_EXTREMES_PER_PAIR_METRIC_CH3, "mean_votes"),
    pairs_pdf_ch2.nlargest(N_EXTREMES_PER_PAIR_METRIC_CH3, "mean_rating"),
    pairs_pdf_ch2.nsmallest(N_EXTREMES_PER_PAIR_METRIC_CH3, "mean_rating"),
    largest_pairs_ch2
]).drop_duplicates("accord_pair").copy()
pair_labels_ch2["is_largest"] = pair_labels_ch2["accord_pair"].isin(largest_pairs_ch2["accord_pair"])
pair_labels_ch2["vote_extremeness"] = (
    (pair_labels_ch2["mean_votes"]-popularity_median_pairs_ch2).abs()/max(abs(popularity_median_pairs_ch2),1)
)
pair_labels_ch2["rating_extremeness"] = (pair_labels_ch2["mean_rating"]-rating_median_pairs_ch2).abs()
pair_labels_ch2 = pair_labels_ch2.sort_values(
    ["is_largest","vote_extremeness","rating_extremeness"],
    ascending=[False,False,False]
).head(MAX_PAIR_LABELS_CH3)

print(f"Eligible dyads plotted: {len(pairs_pdf_ch2):,}")
print(f"Dyads selected for labels: {len(pair_labels_ch2):,}")

fig_3_2_dyads, ax = plt.subplots(figsize=(16,11), dpi=180)
fig_3_2_dyads.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])
for segment_name in segment_order_pairs_ch2:
    subset = pairs_pdf_ch2[pairs_pdf_ch2["segment"] == segment_name]
    if not subset.empty:
        ax.scatter(subset["mean_votes"], subset["mean_rating"], s=subset["bubble_size"],
                   color=segment_colors_pairs_ch2[segment_name], alpha=0.68,
                   edgecolors=BOOK_STYLE["background"], linewidths=0.7,
                   label=segment_name, zorder=2)

ax.axvline(popularity_median_pairs_ch2, color=BOOK_STYLE["muted"], linestyle="--", linewidth=1.1, alpha=0.9)
ax.axhline(rating_median_pairs_ch2, color=BOOK_STYLE["muted"], linestyle="--", linewidth=1.1, alpha=0.9)
for _, row in pair_labels_ch2.iterrows():
    ax.annotate(
        f"{row['accord_pair']}\n(n={int(row['fragrance_count']):,})",
        xy=(row["mean_votes"],row["mean_rating"]), xytext=(7,7), textcoords="offset points",
        fontsize=BOOK_SIZES["footer_pt"], color=BOOK_STYLE["text"], ha="left", va="bottom",
        bbox=dict(boxstyle="round,pad=0.28", facecolor=BOOK_STYLE["background"],
                  edgecolor=BOOK_STYLE["rule"], alpha=0.94),
        arrowprops=dict(arrowstyle="-", color=BOOK_STYLE["muted"], linewidth=0.7, alpha=0.75),
        zorder=5
    )
ax.set_xlabel("Mean community votes per fragrance", fontsize=BOOK_SIZES["category_pt"],
              color=BOOK_STYLE["muted"], labelpad=10)
ax.set_ylabel("Mean fragrance rating", fontsize=BOOK_SIZES["category_pt"],
              color=BOOK_STYLE["muted"], labelpad=10)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda y, _: f"{y:.2f}"))
ax.tick_params(axis="both", length=0, labelsize=BOOK_SIZES["footer_pt"], pad=7)
ax.grid(axis="both", color=BOOK_STYLE["grid"], linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)

pair_segment_legend_ch2 = ax.legend(
    title="Relative segment", loc="upper left", frameon=True,
    facecolor=BOOK_STYLE["background"], edgecolor=BOOK_STYLE["rule"],
    fontsize=BOOK_SIZES["footer_pt"], title_fontsize=BOOK_SIZES["footer_pt"],
    labelcolor=BOOK_STYLE["text"]
)
pair_segment_legend_ch2.get_title().set_color(BOOK_STYLE["muted"])
ref_pair_counts_ch2 = sorted(set([int(pair_counts_ch2.min()), int(pair_counts_ch2.median()), int(pair_counts_ch2.max())]))
pair_size_handles_ch2 = []
for count in ref_pair_counts_ch2:
    size = min_pair_bubble_size_ch2 if pair_counts_ch2.max()==pair_counts_ch2.min() else (
        min_pair_bubble_size_ch2
        + ((count-pair_counts_ch2.min())/(pair_counts_ch2.max()-pair_counts_ch2.min()))**0.5
        * (max_pair_bubble_size_ch2-min_pair_bubble_size_ch2)
    )
    pair_size_handles_ch2.append(ax.scatter([],[],s=size,color=BOOK_STYLE["muted"],alpha=0.55,
        edgecolors=BOOK_STYLE["background"],linewidths=0.7,label=f"{count:,} fragrances"))
pair_size_legend_ch2 = ax.legend(
    handles=pair_size_handles_ch2, title="Bubble size", loc="lower right",
    frameon=True, facecolor=BOOK_STYLE["background"], edgecolor=BOOK_STYLE["rule"],
    fontsize=BOOK_SIZES["footer_pt"], title_fontsize=BOOK_SIZES["footer_pt"],
    labelcolor=BOOK_STYLE["text"]
)
pair_size_legend_ch2.get_title().set_color(BOOK_STYLE["muted"])
ax.add_artist(pair_segment_legend_ch2)

fig_3_2_dyads.subplots_adjust(left=0.10,right=0.97,
    top=1-BOOK_FIGURE_CHART_TOP_IN/fig_3_2_dyads.get_figheight(),bottom=0.11)
add_book_figure_header(
    fig_3_2_dyads,
    "Ordered Accord Dyads: Community Attention and Mean Rating",
    "All eligible dyads  |  Bubble size: number of fragrances  |  Dashed lines: medians across eligible dyads",
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)
fig_3_2_dyads.savefig(CHAPTER_2_OUTPUT_DIR/"figure_2_3_dyad_attention_rating_segmentation.png",
    dpi=300,bbox_inches=None,facecolor=BOOK_STYLE["background"])
show_book_matplotlib(fig_3_2_dyads)
plt.close(fig_3_2_dyads)
print(f"Saved: {CHAPTER_2_OUTPUT_DIR/'figure_2_3_dyad_attention_rating_segmentation.png'}")

## 2.4 Figures 2.3a–2.3d — Highest-Rated Share Within Each Dyad Segment

Each chart shows the 20 dyads within one segment with the highest share of fragrances rated 4.5 or above. This metric complements mean rating; it is not interchangeable with it. Bar labels include the percentage and the number of fragrances in the dyad.

In [ ]:
# ============================================================
# FIGURES 2.3a–2.3d — TOP DYADS BY HIGH-RATING SHARE
# ============================================================

def plot_pair_segment_ch2(segment_name, top_n=20):
    plot_df = (
        pairs_pdf_ch2[pairs_pdf_ch2["segment"] == segment_name]
        .nlargest(top_n,"share_rated_4_5_plus")
        .sort_values("share_rated_4_5_plus",ascending=True).copy()
    )
    if plot_df.empty:
        print(f"No dyads found for segment: {segment_name}")
        return

    fig_height=max(8,len(plot_df)*0.43+2.5)
    fig,ax=plt.subplots(figsize=(15,fig_height),dpi=180)
    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])
    y=np.arange(len(plot_df))
    bars=ax.barh(y,plot_df["share_rated_4_5_plus"],
        color=segment_colors_pairs_ch2[segment_name],edgecolor="none",height=0.68,zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["accord_pair"],fontsize=BOOK_SIZES["category_pt"],color=BOOK_STYLE["text"])
    max_value=float(plot_df["share_rated_4_5_plus"].max())
    for bar,(_,row) in zip(bars,plot_df.iterrows()):
        value=float(row["share_rated_4_5_plus"])
        ax.text(value+max(value*0.012,0.001),bar.get_y()+bar.get_height()/2,
            f"{value:.1%}  (n={int(row['fragrance_count']):,})",
            va="center",ha="left",fontsize=BOOK_SIZES["footer_pt"],color=BOOK_STYLE["text"])
    ax.set_xlim(0,max_value*1.42 if max_value>0 else 1)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("Share of fragrances rated 4.5+",fontsize=BOOK_SIZES["category_pt"],
        color=BOOK_STYLE["muted"],labelpad=10)
    ax.set_ylabel("")
    ax.tick_params(axis="both",length=0,labelsize=BOOK_SIZES["footer_pt"],pad=7)
    ax.grid(axis="x",color=BOOK_STYLE["grid"],linewidth=0.8,zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)
    apply_book_bar_layout(fig,bottom=0.08,right=0.96)
    add_book_figure_header(fig,
        f"{segment_name}: Dyads with the Highest 4.5+ Share",
        "Top 20 ordered dyads by share of fragrances rated 4.5 or above",
        section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2")
    filename=segment_name.lower().replace(" ","-")+"-dyads-ch2.png"
    fig.savefig(CHAPTER_2_OUTPUT_DIR/filename,dpi=300,bbox_inches=None,
        facecolor=BOOK_STYLE["background"])
    show_book_matplotlib(fig)
    plt.close(fig)
    print(f"Saved: {CHAPTER_2_OUTPUT_DIR/filename}")

for segment_name in segment_order_pairs_ch2:
    plot_pair_segment_ch2(segment_name,top_n=20)

## Ordered Accord Triads

This section prepares publication-ready visualizations of ordered three-accord fragrance profiles. It follows the source ordering and aggregation logic established in Notebook 10 while applying the shared ivory editorial design system defined in Notebook 14.

**Analytical conventions**
- The first, second and third positions are preserved; triads with different accord order remain distinct.
- The analysis includes rated fragrances with valid accord data and non-null first, second and third accords.
- The minimum support threshold is 50 fragrances per ordered triad.
- Mean community votes per fragrance are used as an engagement proxy, not as sales or market share.
- The four segments are assigned using the medians of mean votes and mean ratings across all eligible triads.
- The 35 bubble-chart annotations are selected for readability only; all eligible triads remain in the chart and segmentation.
- The share of fragrances rated 4.5+ is a separate descriptive measure from mean rating.

## 2.5 Triad aggregation

Using the shared rated-fragrance dataset prepared in Section 2.0, aggregate the first three accord positions into ordered triads. Apply the same metrics and minimum-support rule as Notebook 10.

In [ ]:
# ============================================================
# CHAPTER 2 — ORDERED TRIAD AGGREGATION
# Shared fragrance-level source is prepared before the dyad section.
# ============================================================

# Aggregate ordered triads using the same metrics as Notebook 10.
accord_triad_stats_ch2 = (
    rating_accords_ch2
    .filter(
        F.col("dominant_accord").isNotNull()
        & F.col("second_accord").isNotNull()
        & F.col("third_accord").isNotNull()
        & F.col("rating_value").isNotNull()
    )
    .groupBy(
        "dominant_accord",
        "second_accord",
        "third_accord"
    )
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.sum("rating_count").alias("total_votes"),
        F.avg("rating_count").alias("mean_votes"),
        F.expr("percentile_approx(rating_count, 0.5)").alias("median_votes"),
        F.avg("rating_value").alias("mean_rating"),
        F.expr("percentile_approx(rating_value, 0.5)").alias("median_rating"),
        F.avg(
            F.when(F.col("rating_value") >= 4.5, 1.0).otherwise(0.0)
        ).alias("share_rated_4_5_plus")
    )
    .withColumn(
        "accord_triad",
        F.concat_ws(
            " + ",
            F.initcap("dominant_accord"),
            F.initcap("second_accord"),
            F.initcap("third_accord")
        )
    )
)

MIN_TRIAD_FRAGRANCES_CH3 = 50

eligible_triads_ch2 = accord_triad_stats_ch2.filter(
    F.col("fragrance_count") >= MIN_TRIAD_FRAGRANCES_CH3
)

print(f"Rated fragrances with valid accord data: {rating_accords_ch2.count():,}")
print(f"Distinct fragrance IDs: {rating_accords_ch2.select('id').distinct().count():,}")
print(f"Unique ordered triads before support filter: {accord_triad_stats_ch2.count():,}")
print(f"Eligible ordered triads (n >= {MIN_TRIAD_FRAGRANCES_CH3}): {eligible_triads_ch2.count():,}")

display(
    eligible_triads_ch2
    .orderBy(F.desc("fragrance_count"), F.desc("mean_rating"))
    .select(
        "accord_triad",
        "fragrance_count",
        "mean_votes",
        "mean_rating",
        "share_rated_4_5_plus"
    )
    .limit(20)
)

## 2.6 Table 2.2 and triad segmentation validation

Validate the fragrance-level uniqueness and eligible triad population before generating publication figures. Segment summary means are calculated across triads, with each triad contributing equally; they are not fragrance-weighted averages.

In [ ]:
# ============================================================
# CHAPTER 2 — VALIDATION AND SEGMENT SUMMARY
# ============================================================

# Source-row uniqueness was validated in the shared Chapter 2 setup cell.
eligible_triad_count_ch2 = eligible_triads_ch2.count()
eligible_fragrance_assignments_ch2 = (
    eligible_triads_ch2
    .agg(F.sum("fragrance_count").alias("n"))
    .first()["n"]
)

assert eligible_triad_count_ch2 == 159, (
    f"Expected 159 eligible triads; found {eligible_triad_count_ch2:,}. "
    "Review the source snapshot and eligibility rules before publication."
)
assert eligible_fragrance_assignments_ch2 == 16_073, (
    f"Expected 16,073 fragrance assignments; found "
    f"{eligible_fragrance_assignments_ch2:,}."
)

print("Chapter 2 validation checks passed.")
print(f"Unique rated fragrance records: {source_rows_ch2:,}")
print(f"Eligible ordered triads: {eligible_triad_count_ch2:,}")
print(f"Fragrances represented in eligible triads: {eligible_fragrance_assignments_ch2:,}")

# Prepare one shared pandas dataset for the Chapter 2 figures.
triads_pdf_ch2 = eligible_triads_ch2.toPandas().copy()

for metric in [
    "fragrance_count",
    "mean_votes",
    "mean_rating",
    "share_rated_4_5_plus"
]:
    triads_pdf_ch2[metric] = pd.to_numeric(
        triads_pdf_ch2[metric],
        errors="coerce"
    )

triads_pdf_ch2 = triads_pdf_ch2.dropna(
    subset=[
        "accord_triad",
        "fragrance_count",
        "mean_votes",
        "mean_rating",
        "share_rated_4_5_plus"
    ]
).copy()

# Median thresholds are calculated across all eligible triads.
popularity_median_ch2 = triads_pdf_ch2["mean_votes"].median()
rating_median_ch2 = triads_pdf_ch2["mean_rating"].median()

def assign_triad_segment_ch2(row):
    high_attention = row["mean_votes"] >= popularity_median_ch2
    high_rating = row["mean_rating"] >= rating_median_ch2

    if high_attention and high_rating:
        return "Standout Profiles"
    elif not high_attention and high_rating:
        return "Hidden Favourites"
    elif high_attention and not high_rating:
        return "Widely Explored"
    else:
        return "Quiet Profiles"

triads_pdf_ch2["segment"] = triads_pdf_ch2.apply(
    assign_triad_segment_ch2,
    axis=1
)

segment_order_ch2 = [
    "Standout Profiles",
    "Hidden Favourites",
    "Widely Explored",
    "Quiet Profiles"
]

# Reuse the muted colours already defined in Notebook 14's global palette.
segment_colors_ch2 = {
    "Standout Profiles": BOOK_STYLE["accent"],
    "Hidden Favourites": BOOK_STYLE["women"],
    "Widely Explored": BOOK_STYLE["men"],
    "Quiet Profiles": BOOK_STYLE["unisex"]
}

print(f"Median mean community votes per fragrance: {popularity_median_ch2:,.2f}")
print(f"Median mean fragrance rating: {rating_median_ch2:.3f}")

segment_summary_ch2 = (
    triads_pdf_ch2
    .groupby("segment", observed=False)
    .agg(
        triad_count=("accord_triad", "count"),
        fragrance_assignments=("fragrance_count", "sum"),
        mean_votes_per_triad=("mean_votes", "mean"),
        mean_rating_per_triad=("mean_rating", "mean"),
        mean_triad_share_rated_4_5_plus=("share_rated_4_5_plus", "mean")
    )
    .reindex(segment_order_ch2)
    .reset_index()
)

segment_summary_display_ch2 = segment_summary_ch2.copy()
segment_summary_display_ch2["mean_votes_per_triad"] = (
    segment_summary_display_ch2["mean_votes_per_triad"].round(1)
)
segment_summary_display_ch2["mean_rating_per_triad"] = (
    segment_summary_display_ch2["mean_rating_per_triad"].round(3)
)
segment_summary_display_ch2["mean_triad_share_rated_4_5_plus"] = (
    segment_summary_display_ch2["mean_triad_share_rated_4_5_plus"].map(
        lambda value: f"{value:.1%}" if pd.notna(value) else ""
    )
)

triad_segment_table_ch2 = make_book_table(
    segment_summary_display_ch2.rename(columns={
        "segment": "Segment",
        "triad_count": "Ordered triads",
        "fragrance_assignments": "Fragrances represented",
        "mean_votes_per_triad": "Mean votes per fragrance",
        "mean_rating_per_triad": "Mean rating",
        "mean_triad_share_rated_4_5_plus": "Mean triad share rated 4.5+"
    }),
    title="Ordered Triad Segmentation Summary",
    subtitle=(
        "Segment means are unweighted across triads; fragrance counts are "
        "the sum of the group-level triad counts."
    )
)
show_book_table(triad_segment_table_ch2)

display(
    triads_pdf_ch2[
        [
            "accord_triad",
            "fragrance_count",
            "mean_votes",
            "mean_rating",
            "share_rated_4_5_plus",
            "segment"
        ]
    ].sort_values("fragrance_count", ascending=False).head(20)
)

## 2.7 Figure 2.4 — Most Frequent Ordered Accord Triads

The chart ranks the 20 most frequent ordered triads by the number of fragrances assigned to each combination. Accord order is retained in every label.

In [ ]:
# ============================================================
# FIGURE 2.4 — MOST FREQUENT ORDERED TRIADS
# ============================================================

def format_triad_label_ch2(value):
    parts = str(value).split(" + ")
    if len(parts) == 3:
        return f"{parts[0]} + {parts[1]}\n{parts[2]}"
    return str(value)

def plot_triad_ranking_ch2(
    metric,
    title,
    subtitle,
    filename,
    bar_color,
    value_format="number",
    top_n=20
):
    plot_df = (
        triads_pdf_ch2
        .dropna(subset=[metric])
        .nlargest(top_n, metric)
        .sort_values(metric, ascending=True)
        .copy()
    )

    plot_df["display_label"] = plot_df["accord_triad"].apply(
        format_triad_label_ch2
    )

    fig_height = max(8, len(plot_df) * 0.43 + 2.5)
    fig, ax = plt.subplots(figsize=(15, fig_height), dpi=180)
    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])

    y_positions = np.arange(len(plot_df))
    bars = ax.barh(
        y_positions,
        plot_df[metric],
        color=bar_color,
        edgecolor="none",
        height=0.68,
        zorder=3
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        plot_df["display_label"],
        fontsize=BOOK_SIZES["footer_pt"],
        color=BOOK_STYLE["text"]
    )

    max_value = float(plot_df[metric].max())
    label_offset = max(max_value * 0.012, 0.001)

    for bar, (_, row) in zip(bars, plot_df.iterrows()):
        value = float(row[metric])
        if value_format == "rating":
            label = f"{value:.3f}"
        elif value_format == "percentage":
            label = f"{value:.1%}"
        elif value_format == "votes":
            label = f"{value:,.1f}"
        else:
            label = f"{value:,.0f}"

        ax.text(
            value + label_offset,
            bar.get_y() + bar.get_height() / 2,
            label,
            va="center",
            ha="left",
            fontsize=BOOK_SIZES["footer_pt"],
            color=BOOK_STYLE["text"]
        )

    ax.set_xlim(0, max_value * 1.16 if max_value > 0 else 1)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="both", length=0, labelsize=BOOK_SIZES["footer_pt"], pad=7)
    ax.grid(axis="x", color=BOOK_STYLE["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

    if value_format == "percentage":
        ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    elif value_format == "votes":
        ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    elif value_format == "rating":
        ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.2f}"))
    else:
        ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    for spine in ax.spines.values():
        spine.set_visible(False)

    apply_book_bar_layout(fig, bottom=0.08, right=0.96)
    add_book_figure_header(
        fig,
        title,
        subtitle,
        section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
    )

    fig.savefig(
        CHAPTER_2_OUTPUT_DIR / filename,
        dpi=300,
        bbox_inches=None,
        facecolor=BOOK_STYLE["background"]
    )
    show_book_matplotlib(fig)
    plt.close(fig)
    print(f"Saved: {CHAPTER_2_OUTPUT_DIR / filename}")

plot_triad_ranking_ch2(
    metric="fragrance_count",
    title="The Most Frequent Ordered Accord Triads",
    subtitle="Twenty most frequent combinations; n denotes the number of fragrances",
    filename="figure_2_2_top_20_ordered_triads.png",
    bar_color=BOOK_STYLE["accent"],
    value_format="number"
)

## 2.8 Figure 2.5 — Community Attention and Mean Rating Across Ordered Triads

The bubble chart positions all eligible ordered triads by mean community votes per fragrance and mean fragrance rating. Bubble size represents the number of fragrances in the triad. Dashed reference lines mark the medians calculated across the full eligible population. Labels identify a selected subset of up to 35 triads for readability; they do not define the analytical population.

In [ ]:
# ============================================================
# FIGURE 2.5 — ORDERED TRIAD ATTENTION / RATING SEGMENTATION
# All eligible triads are plotted; 35 selected triads are annotated.
# ============================================================

N_LARGEST_TRIADS_CH3 = 10
N_EXTREMES_PER_METRIC_CH3 = 10
MAX_LABELS_CH3 = 35

min_bubble_size_ch2 = 55
max_bubble_size_ch2 = 950
counts_ch2 = triads_pdf_ch2["fragrance_count"].astype(float)

if counts_ch2.max() == counts_ch2.min():
    triads_pdf_ch2["bubble_size"] = min_bubble_size_ch2
else:
    triads_pdf_ch2["bubble_size"] = (
        min_bubble_size_ch2
        + (
            (counts_ch2 - counts_ch2.min())
            / (counts_ch2.max() - counts_ch2.min())
        ) ** 0.5
        * (max_bubble_size_ch2 - min_bubble_size_ch2)
    )

highest_votes_ch2 = triads_pdf_ch2.nlargest(
    N_EXTREMES_PER_METRIC_CH3, "mean_votes"
)
lowest_votes_ch2 = triads_pdf_ch2.nsmallest(
    N_EXTREMES_PER_METRIC_CH3, "mean_votes"
)
highest_ratings_ch2 = triads_pdf_ch2.nlargest(
    N_EXTREMES_PER_METRIC_CH3, "mean_rating"
)
lowest_ratings_ch2 = triads_pdf_ch2.nsmallest(
    N_EXTREMES_PER_METRIC_CH3, "mean_rating"
)
largest_triads_ch2 = triads_pdf_ch2.nlargest(
    N_LARGEST_TRIADS_CH3, "fragrance_count"
)

label_candidates_ch2 = pd.concat([
    highest_votes_ch2,
    lowest_votes_ch2,
    highest_ratings_ch2,
    lowest_ratings_ch2,
    largest_triads_ch2
]).drop_duplicates(subset=["accord_triad"]).copy()

label_candidates_ch2["is_largest"] = (
    label_candidates_ch2["accord_triad"].isin(
        largest_triads_ch2["accord_triad"]
    )
)
label_candidates_ch2["vote_extremeness"] = (
    (label_candidates_ch2["mean_votes"] - popularity_median_ch2).abs()
    / max(abs(popularity_median_ch2), 1)
)
label_candidates_ch2["rating_extremeness"] = (
    (label_candidates_ch2["mean_rating"] - rating_median_ch2).abs()
)

label_candidates_ch2 = (
    label_candidates_ch2
    .sort_values(
        by=["is_largest", "vote_extremeness", "rating_extremeness"],
        ascending=[False, False, False]
    )
    .head(MAX_LABELS_CH3)
)

print(f"Eligible triads plotted: {len(triads_pdf_ch2):,}")
print(f"Triads selected for labels: {len(label_candidates_ch2):,}")

fig_3_2, ax = plt.subplots(figsize=(16, 11), dpi=180)
fig_3_2.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

for segment_name in segment_order_ch2:
    subset = triads_pdf_ch2[
        triads_pdf_ch2["segment"] == segment_name
    ]
    if subset.empty:
        continue

    ax.scatter(
        subset["mean_votes"],
        subset["mean_rating"],
        s=subset["bubble_size"],
        color=segment_colors_ch2[segment_name],
        alpha=0.68,
        edgecolors=BOOK_STYLE["background"],
        linewidths=0.7,
        label=segment_name,
        zorder=2
    )

ax.axvline(
    popularity_median_ch2,
    color=BOOK_STYLE["muted"],
    linestyle="--",
    linewidth=1.1,
    alpha=0.9,
    zorder=1
)
ax.axhline(
    rating_median_ch2,
    color=BOOK_STYLE["muted"],
    linestyle="--",
    linewidth=1.1,
    alpha=0.9,
    zorder=1
)

for _, row in label_candidates_ch2.iterrows():
    label_text = (
        f"{format_triad_label_ch2(row['accord_triad'])}"
        f"\n(n={int(row['fragrance_count']):,})"
    )
    ax.annotate(
        label_text,
        xy=(row["mean_votes"], row["mean_rating"]),
        xytext=(7, 7),
        textcoords="offset points",
        fontsize=BOOK_SIZES["footer_pt"],
        color=BOOK_STYLE["text"],
        ha="left",
        va="bottom",
        bbox=dict(
            boxstyle="round,pad=0.28",
            facecolor=BOOK_STYLE["background"],
            edgecolor=BOOK_STYLE["rule"],
            alpha=0.94
        ),
        arrowprops=dict(
            arrowstyle="-",
            color=BOOK_STYLE["muted"],
            linewidth=0.7,
            alpha=0.75
        ),
        zorder=5
    )

ax.set_xlabel(
    "Mean community votes per fragrance",
    fontsize=BOOK_SIZES["category_pt"],
    color=BOOK_STYLE["muted"],
    labelpad=10
)
ax.set_ylabel(
    "Mean fragrance rating",
    fontsize=BOOK_SIZES["category_pt"],
    color=BOOK_STYLE["muted"],
    labelpad=10
)
ax.xaxis.set_major_formatter(
    mtick.FuncFormatter(lambda x, _: f"{x:,.0f}")
)
ax.yaxis.set_major_formatter(
    mtick.FuncFormatter(lambda y, _: f"{y:.2f}")
)
ax.tick_params(axis="both", length=0, labelsize=BOOK_SIZES["footer_pt"], pad=7)
ax.grid(axis="both", color=BOOK_STYLE["grid"], linewidth=0.7, zorder=0)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

segment_legend = ax.legend(
    title="Relative segment",
    loc="upper left",
    frameon=True,
    facecolor=BOOK_STYLE["background"],
    edgecolor=BOOK_STYLE["rule"],
    fontsize=BOOK_SIZES["footer_pt"],
    title_fontsize=BOOK_SIZES["footer_pt"],
    labelcolor=BOOK_STYLE["text"]
)
segment_legend.get_title().set_color(BOOK_STYLE["muted"])

reference_counts_ch2 = sorted(set([
    int(counts_ch2.min()),
    int(counts_ch2.median()),
    int(counts_ch2.max())
]))
size_handles_ch2 = []

for count in reference_counts_ch2:
    if counts_ch2.max() == counts_ch2.min():
        reference_size = min_bubble_size_ch2
    else:
        reference_size = (
            min_bubble_size_ch2
            + (
                (count - counts_ch2.min())
                / (counts_ch2.max() - counts_ch2.min())
            ) ** 0.5
            * (max_bubble_size_ch2 - min_bubble_size_ch2)
        )

    size_handles_ch2.append(
        ax.scatter(
            [], [], s=reference_size,
            color=BOOK_STYLE["muted"], alpha=0.55,
            edgecolors=BOOK_STYLE["background"],
            linewidths=0.7,
            label=f"{count:,} fragrances"
        )
    )

size_legend_ch2 = ax.legend(
    handles=size_handles_ch2,
    title="Bubble size",
    loc="lower right",
    frameon=True,
    facecolor=BOOK_STYLE["background"],
    edgecolor=BOOK_STYLE["rule"],
    fontsize=BOOK_SIZES["footer_pt"],
    title_fontsize=BOOK_SIZES["footer_pt"],
    labelcolor=BOOK_STYLE["text"]
)
size_legend_ch2.get_title().set_color(BOOK_STYLE["muted"])
ax.add_artist(segment_legend)

fig_3_2.subplots_adjust(
    left=0.10,
    right=0.97,
    top=1 - BOOK_FIGURE_CHART_TOP_IN / fig_3_2.get_figheight(),
    bottom=0.11
)
add_book_figure_header(
    fig_3_2,
    "Ordered Accord Triads: Community Attention and Mean Rating",
    (
        "All eligible triads  |  Bubble size: number of fragrances  |  "
        "Dashed lines: medians across eligible triads"
    ),
    section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
)

fig_3_2.savefig(
    CHAPTER_2_OUTPUT_DIR / "figure_2_4_triad_attention_rating_segmentation.png",
    dpi=300,
    bbox_inches=None,
    facecolor=BOOK_STYLE["background"]
)
show_book_matplotlib(fig_3_2)
plt.close(fig_3_2)
print(f"Saved: {CHAPTER_2_OUTPUT_DIR / 'figure_2_4_triad_attention_rating_segmentation.png'}")

## 2.9 Figures 2.6a–2.6d — Highest-Rated Share Within Each Triad Segment

Each chart shows the 20 triads within one segment with the highest share of fragrances rated 4.5 or above. The metric complements mean rating; it is not interchangeable with it. Bar labels include the percentage and the number of fragrances in the triad.

In [ ]:
# ============================================================
# FIGURES 2.6a–2.6d — TOP TRIADS BY HIGH-RATING SHARE, BY SEGMENT
# ============================================================

def plot_triad_segment_ch2(segment_name, top_n=20):
    plot_df = (
        triads_pdf_ch2[
            triads_pdf_ch2["segment"] == segment_name
        ]
        .nlargest(top_n, "share_rated_4_5_plus")
        .sort_values("share_rated_4_5_plus", ascending=True)
        .copy()
    )

    if plot_df.empty:
        print(f"No triads found for segment: {segment_name}")
        return

    plot_df["display_label"] = plot_df["accord_triad"].apply(
        format_triad_label_ch2
    )

    fig_height = max(8, len(plot_df) * 0.43 + 2.5)
    fig, ax = plt.subplots(figsize=(15, fig_height), dpi=180)
    fig.patch.set_facecolor(BOOK_STYLE["background"])
    ax.set_facecolor(BOOK_STYLE["background"])

    y_positions = np.arange(len(plot_df))
    bars = ax.barh(
        y_positions,
        plot_df["share_rated_4_5_plus"],
        color=segment_colors_ch2[segment_name],
        edgecolor="none",
        height=0.68,
        zorder=3
    )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        plot_df["display_label"],
        fontsize=BOOK_SIZES["footer_pt"],
        color=BOOK_STYLE["text"]
    )

    max_value = float(plot_df["share_rated_4_5_plus"].max())
    label_offset = max(max_value * 0.012, 0.001)

    for bar, (_, row) in zip(bars, plot_df.iterrows()):
        value = float(row["share_rated_4_5_plus"])
        ax.text(
            value + label_offset,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.1%}  (n={int(row['fragrance_count']):,})",
            va="center",
            ha="left",
            fontsize=BOOK_SIZES["footer_pt"],
            color=BOOK_STYLE["text"]
        )

    ax.set_xlim(0, max_value * 1.42 if max_value > 0 else 1)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel(
        "Share of fragrances rated 4.5+",
        fontsize=BOOK_SIZES["category_pt"],
        color=BOOK_STYLE["muted"],
        labelpad=10
    )
    ax.set_ylabel("")
    ax.tick_params(axis="both", length=0, labelsize=BOOK_SIZES["footer_pt"], pad=7)
    ax.grid(axis="x", color=BOOK_STYLE["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(False)

    apply_book_bar_layout(fig, bottom=0.08, right=0.96)
    add_book_figure_header(
        fig,
        f"{segment_name}: Triads with the Highest 4.5+ Share",
        "Top 20 ordered triads by share of fragrances rated 4.5 or above",
        section="FRAGRANCE INTELLIGENCE  /  CHAPTER 2"
    )

    filename = (
        segment_name.lower().replace(" ", "-") + "-triads-ch2.png"
    )
    fig.savefig(
        CHAPTER_2_OUTPUT_DIR / filename,
        dpi=300,
        bbox_inches=None,
        facecolor=BOOK_STYLE["background"]
    )
    show_book_matplotlib(fig)
    plt.close(fig)
    print(f"Saved: {CHAPTER_2_OUTPUT_DIR / filename}")

for segment_name in segment_order_ch2:
    plot_triad_segment_ch2(segment_name, top_n=20)

## 2.6 Dyad and triad comparison

The following comparison brings the segment summaries together. Segment-level means remain unweighted across combinations; fragrance assignments are summed group-level counts. The dyad and triad thresholds are independent and their eligible populations differ.

In [ ]:
# ============================================================
# TABLE 2.3 — DYAD AND TRIAD SEGMENT COMPARISON
# ============================================================

pair_compare_ch2 = pair_segment_summary_ch2.rename(columns={
    'pair_count': 'dyad_count',
    'fragrance_assignments': 'dyad_fragrance_assignments',
    'mean_votes_per_pair': 'dyad_mean_votes',
    'mean_rating_per_pair': 'dyad_mean_rating',
    'mean_pair_share_rated_4_5_plus': 'dyad_mean_share_4_5_plus'
})

triad_compare_ch2 = segment_summary_ch2.rename(columns={
    'triad_count': 'triad_count',
    'fragrance_assignments': 'triad_fragrance_assignments',
    'mean_votes_per_triad': 'triad_mean_votes',
    'mean_rating_per_triad': 'triad_mean_rating',
    'mean_triad_share_rated_4_5_plus': 'triad_mean_share_4_5_plus'
})

dyad_triad_comparison_ch2 = pair_compare_ch2.merge(
    triad_compare_ch2, on='segment', how='outer'
)
display(dyad_triad_comparison_ch2.round(3))

print('Dyad attention median:', round(popularity_median_pairs_ch2, 3))
print('Dyad rating median:', round(rating_median_pairs_ch2, 4))
print('Triad attention median:', round(popularity_median_ch2, 3))
print('Triad rating median:', round(rating_median_ch2, 4))

## 2.7 Reproducibility and output validation

- Source table: `main.fragrance_analytics.fragrantica_perfumes`.
- Accord positions are preserved in their original order.
- Dyad aggregation uses a minimum support of 30 fragrances; segmentation uses at least 100 (147 eligible dyads in the validated source snapshot).
- Triad segmentation uses at least 50 fragrances (159 eligible triads and 16,073 fragrance assignments in the validated source snapshot).
- Dyad and triad medians are calculated independently.
- Segment summary metrics are unweighted means across combinations.
- Segment-specific charts rank by the share of associated fragrances rated 4.5 or above; this measure does not define segment membership.
- PNG files are saved at 300 dpi.
- Community votes represent platform activity, not sales, market share or unique consumers.

**Output location:** `/Volumes/main/fragrance_analytics/reference_data/visualizations/`.